## Ejercicios de Particionado de Windows
1. Rol más reciente por desarrollador: Usando el archivo roles.csv, obtén para cada desarrollador (dev_id) su rol más reciente según from_date.

2. Diferencia de duración de roles por desarrollador: Usando roles.csv, calcula la duración de cada rol en días (to_date - from_date). Luego, calcula la diferencia de duración de cada rol respecto al rol de menor duración del mismo desarrollador.

In [ ]:
# Cargar datasets
developersDF = spark.read.option("header", "true").csv("developers.csv")
rolesDF = spark.read.option("header", "true").csv("roles.csv")

# Convertir fechas a tipo date
rolesDF = rolesDF.withColumn("from_date", col("from_date").cast("date")) \
                 .withColumn("to_date", col("to_date").cast("date"))

# ===========================
# Ejercicio 1: Rol más reciente por dev_id
# ===========================
window_recent = Window.partitionBy("dev_id").orderBy(col("from_date").desc())

recent_rolesDF = rolesDF.withColumn("rn", row_number().over(window_recent)) \
                        .filter(col("rn") == 1) \
                        .select("dev_id", "role_name", "from_date")

print("=== Rol más reciente por desarrollador ===")
recent_rolesDF.show()

# ===========================
# Ejercicio 2: Diferencia de duración de roles por dev_id
# ===========================
# Calcular duración en días
rolesDF = rolesDF.withColumn("duration_days", datediff(col("to_date"), col("from_date")))

# Window por dev_id para calcular la duración mínima
window_duration = Window.partitionBy("dev_id").orderBy(col("duration_days").asc())

diff_durationDF = rolesDF.withColumn("duration_diff", col("duration_days") - min(col("duration_days")).over(window_duration)) \
                         .select("dev_id", "role_name", "duration_days", "duration_diff")

print("=== Diferencia de duración de roles por desarrollador ===")
diff_durationDF.show()

## Ejercicios UDFs
1. Elige uno de los DF que tenemos, define dos UDF propios y aplícalos (con un withColumn) al DF. Muestra los resultados. 

In [ ]:
# ======================================
# 1. Cargar el CSV
# ======================================
df = (
    spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv("roles.csv")
)

print("=== DataFrame original ===")
df.show()

# ======================================
# 2. Definir UDF #1: es_senior
# ======================================
def f_es_senior(role):
    if role is None:
        return False
    return "senior" in role.lower()

es_senior_udf = udf(f_es_senior, BooleanType())

# ======================================
# 3. Definir UDF #2: antiguedad_anos
# ======================================
def f_antiguedad(from_date):
    if from_date is None:
        return None
    
    try:
        fecha = from_date if isinstance(from_date, datetime) else datetime.strptime(from_date, "%Y-%m-%d")
        return datetime.now().year - fecha.year
    except:
        return None

antiguedad_udf = udf(f_antiguedad, IntegerType())

# ======================================
# 4. Aplicar UDFs con withColumn
# ======================================
df_udf = (
    df.withColumn("es_senior", es_senior_udf(col("role_name")))
      .withColumn("antiguedad", antiguedad_udf(col("from_date")))
)

print("=== DataFrame con UDFs aplicadas ===")
df_udf.show()